# 18 — Production Quality: Packaging, CI, Configuration, and Deployment

Robust software ships with reproducible dependencies, clear entry points, automated checks, safe configuration, and lifecycle controls. Turn notebook knowledge into maintainable project structure.

*Advanced · 90–150 min*

## You will learn
- Design a src-layout Python package
- Describe CI stages including syntax, tests, type checks, and security checks
- Distinguish config and secrets from code
- Prepare an application health and shutdown strategy

## Examples

### 1. Project layout and build metadata

pyproject.toml describes build backends, dependencies, entry points and Python compatibility. Use a src layout to reduce accidental import-from-working-directory problems. The example prints a project structure instead of creating files.

In [ ]:
layout = """service/
  pyproject.toml
  README.md
  src/service/__init__.py
  src/service/core.py
  src/service/cli.py
  tests/test_core.py
"""
print(layout)

**Review:** Why can importing directly from a repository root hide packaging mistakes?

### 2. Configuration validation

Keep secrets in managed secret storage or environment variables rather than public source files. Validate configuration at startup and make invalid settings fail clearly. Avoid logging full configuration dictionaries containing credentials.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Config:
    timeout: float = 5.0
    concurrency: int = 4
    def __post_init__(self):
        if self.timeout <= 0 or self.concurrency < 1:
            raise ValueError("invalid configuration")

config = Config()
print(config.timeout, config.concurrency)

**Review:** What is the difference between a configuration value and a secret credential?

### 3. CI as automated evidence

A CI pipeline should check syntax, formatting, automated tests, and relevant integration cases. Keep the feedback fast and deterministic; longer benchmarks and security scans can run at appropriate intervals. The real repository's workflow validates notebook syntax and execution.

In [ ]:
stages = ["syntax", "unit tests", "integration tests", "package build", "security review"]
for index, stage in enumerate(stages, start=1):
    print(index, stage)

**Review:** Why should tests not depend on live APIs or arbitrary sleep durations?

### 4. Graceful shutdown and cleanup

A service should stop accepting new work, finish or cancel bounded tasks, and close resources cleanly. Signal handling differs across OS and runtime environments; use explicit application lifecycle functions and context managers whenever possible.

In [ ]:
from contextlib import contextmanager

@contextmanager
def managed_resource(events):
    events.append("start")
    try:
        yield object()
    finally:
        events.append("close")

events = []
with managed_resource(events):
    events.append("work")
print(events)
assert events == ["start", "work", "close"]

**Review:** Why is cleanup in finally important during cancellation or unexpected exceptions?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_production_systems_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/18', {1: '0c76f5b33ccfc75d', 2: 'bf96b4107fc546cb', 3: 'b4586e1ff25baf74', 4: 'c8f3dc4eab9cfe80'})

**Exercise 1.** A `pyproject.toml` for an installable CLI with one command.

In [ ]:
import tomllib
text = """
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "notes"
version = "0.1.0"
requires-python = ">=3.11"

[project.scripts]
notes = "notes.cli:main"
"""
data = tomllib.loads(text)
check(1, (data["project"]["name"], data["project"]["scripts"][___]))

**Exercise 2.** A CI matrix that covers two Python versions.

In [ ]:
matrix = {"python-version": ["3.11", ___]}
jobs = [f"test on {v}" for v in matrix["python-version"]]
check(2, len(jobs) >= 2 and all(v.startswith("3.") for v in matrix["python-version"]))

**Exercise 3.** A validated config whose `repr` never shows the secret.

In [ ]:
from dataclasses import dataclass, field

@dataclass(frozen=True)
class Config:
    database_url: str
    api_key: str = field(repr=___)

    def __post_init__(self):
        if not self.database_url.startswith("sqlite"):
            raise ValueError("only sqlite is supported here")

c = Config("sqlite:///app.db", api_key="sk-secret")
check(3, ("sk-secret" in repr(c), c.api_key == "sk-secret"))

**Exercise 4.** Startup, readiness and graceful shutdown as a tiny state machine.

In [ ]:
class Service:
    def __init__(self):
        self.state = "starting"
    def ready(self):
        return self.state == ___
    def start(self):
        self.state = "running"
    def shutdown(self):
        self.state = "draining"
        self.state = "stopped"

s = Service()
states = [s.ready()]
s.start(); states.append(s.ready())
s.shutdown(); states.append((s.ready(), s.state))
check(4, states)

## Common mistakes

- Do not embed access tokens in notebooks or configuration checked into Git.
- CI success is evidence for tested behavior, not proof of all production conditions.
- Application cleanup must handle failures as well as success.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Secrets should come from…

   - **a)** Environment or a secret store at runtime
   - **b)** The source code
   - **c)** Git history

**2.** A readiness check answers…

   - **a)** Can it serve traffic right now (dependencies OK)?
   - **b)** Has it started?
   - **c)** Is the process alive?

**3.** A useful CI pipeline runs…

   - **a)** Only manual tests
   - **b)** Only on release day
   - **c)** Syntax checks and tests on every change

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/18', {1: '17e8994c8384', 2: 'b1ff8b23a6aa', 3: '00a05fd10354'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [19 — Advanced Python Review and Capstone Preparation](19_advanced_review_capstone.ipynb)